# Frozen Text embedding model to export text embeddings

In [ ]:

import wandb
wandb.init(project="morpheus-embeddings", name="frozen_embedding_export", job_type="dataset-generation")

In [ ]:
# load input data
import zipfile
dataset_url = "https://huggingface.co/datasets/zjunlp/Mol-Instructions/resolve/main/data/Molecule-oriented_Instructions.zip"
zip_file = "Molecule-oriented_Instructions.zip"
extract_dir = "./data_mol_instruct"

# Download and unzip
if not os.path.exists(extract_dir):
    print(f"Downloading {dataset_url}...")
    os.system(f"wget -q {dataset_url} -O {zip_file}")

    print(f"Extracting {zip_file}...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction complete.")
else:
    print("Dataset already downloaded and extracted.")
    

In [ ]:
import pandas as pd
import json
import os
import wandb  

# Load JSON
json_path = os.path.join("./", "data_mol_instruct/Molecule-oriented_Instructions/description_guided_molecule_design.json")
with open(json_path, 'r') as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

def format_prompt(row):
    instruction = row.get('instruction', '')
    inp = row.get('input', '')
    if inp:
        return f"{instruction}\nInput: {inp}"
    return instruction

df['prompt'] = df.apply(format_prompt, axis=1)
df['response'] = df['output']

# Split into train and validation
df['split'] = df['metadata'].apply(lambda m: m['split'])
train_df_ = df[df['split'] == 'train'][['prompt', 'response']]
train_df = train_df_.sample(frac=0.9, random_state=42)
val_df = train_df_.drop(train_df.index)
test_df = df[df['split'] == 'test'][['prompt', 'response']]

# Save to CSV
train_csv_path = os.path.join("./", "train.csv")
val_csv_path = os.path.join("./", "val.csv")
test_csv_path = os.path.join("./", "test.csv")

train_df.to_csv(train_csv_path, index=False)
val_df.to_csv(val_csv_path, index=False)
test_df.to_csv(test_csv_path, index=False)

# Log dataset
artifact = wandb.Artifact("morpheus-datasets", type="dataset")
artifact.add_file(train_csv_path)
artifact.add_file(val_csv_path)
artifact.add_file(test_csv_path)
wandb.log_artifact(artifact)

print(f"Saved train csv to {train_csv_path}")
print(f"Saved val csv to {val_csv_path}")

print(df['split'].value_counts())
print(len(train_df), len(val_df), len(test_df))

In [ ]:
# Load Model
from transformers import AutoTokenizer, AutoModel
def load_text_encoder(model_name: str, device: torch.device):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    return tokenizer, model


In [ ]:
# Export data
import pandas as pd
import numpy as np
import os
from datetime import datetime, timezone
from pathlib import Path
import wandb
from tqdm.notebook import tqdm  

SPLITS = {
    "train": "train.csv",
    "val": "val.csv",
    "test": "test.csv"
}
PROMPT_COLUMN = "prompt"
MODEL_NAME = "BAAI/bge-large-en-v1.5"
BATCH_SIZE = 128
MAX_LENGTH = 256
DEVICE =  torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = Path("outputs/embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load model
text_tokenizer, text_encoder = load_text_encoder(MODEL_NAME, DEVICE)
text_encoder.eval()

def encode_text_batch_full_seq(texts, tokenizer, model, device, max_length):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
        return out.last_hidden_state.detach().cpu().numpy(), enc['attention_mask'].cpu().numpy()

for split, csv_path in SPLITS.items():
    print(f"Processing split: {split}")
    df = pd.read_csv(csv_path)
    valid = df[PROMPT_COLUMN].dropna().astype(str)
    valid = valid[valid.str.strip() != ""]
    texts = valid.tolist()
    all_embeddings = []
    all_attention_masks = []
    for s in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"Encoding {split}"):
        emb, attn_mask = encode_text_batch_full_seq(texts[s:s+BATCH_SIZE], text_tokenizer, text_encoder, DEVICE, MAX_LENGTH)
        all_embeddings.append(emb)
        all_attention_masks.append(attn_mask)
    # Concatenate all batches
    emb_matrix = np.concatenate(all_embeddings, axis=0).astype(np.float32)  # (N, seq_len, hidden_dim)
    attn_matrix = np.concatenate(all_attention_masks, axis=0).astype(np.int32)  # (N, seq_len)
    num_samples, seq_len, embed_dim = emb_matrix.shape
    out = pd.DataFrame()
    out['source_index'] = valid.index
    out['text'] = valid
    out['text_length'] = out['text'].str.len()
    out['model_name'] = MODEL_NAME
    out['embedding_dim'] = embed_dim
    out['seq_len'] = seq_len
    out['generated_at_utc'] = datetime.now(timezone.utc).isoformat()
    out['embeddings'] = [emb_matrix[i].tolist() for i in range(num_samples)]
    out['attention_mask'] = [attn_matrix[i].tolist() for i in range(num_samples)]
    out_path = OUT_DIR / f"{split}_prompt_fullseq_embeddings.parquet"
    try:
        out.to_parquet(out_path, index=False, engine='pyarrow')
    except Exception:
        out.to_parquet(out_path, index=False, engine='fastparquet')
    print(f"Saved {split} full sequence prompt embeddings to:", out_path)
    # Log embedding parquet as wandb artifact
    emb_artifact = wandb.Artifact(f"{split}_prompt_fullseq_embeddings", type="embedding")
    emb_artifact.add_file(str(out_path))
    wandb.log_artifact(emb_artifact)
    # Log metrics for this split
    wandb.log({f"{split}_num_samples": num_samples, f"{split}_embedding_dim": embed_dim, f"{split}_seq_len": seq_len})

In [ ]:

wandb.finish()